In [5]:
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, f1_score

# ---------------------------------------------------------
# 1-3. CARGAR DATOS Y SEPARAR (Igual que antes)
# ---------------------------------------------------------
print("Cargando datasets...")
train = pd.read_csv('train_200000.csv', header=None)
val = pd.read_csv('validation.csv', header=None)

y_train = train.iloc[:, 0]
X_train = train.iloc[:, 1:29]

y_val = val.iloc[:, 0]
X_val = val.iloc[:, 1:29]

if 'observation_id' in val.columns:
    val_ids = val['observation_id']
    X_val = X_val.drop(columns=['observation_id'], errors='ignore')
    X_train = X_train.drop(columns=['observation_id'], errors='ignore')
else:
    val_ids = val.index

# ---------------------------------------------------------
# 4-5. PREPROCESSING ESTRICTO EN TRAIN
# ---------------------------------------------------------
print("Aplicando preprocesamiento...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# ---------------------------------------------------------
# NUEVO: BÚSQUEDA DE HIPERPARÁMETROS (GridSearchCV)
# ---------------------------------------------------------
print("Iniciando búsqueda de hiperparámetros (esto tomará un par de minutos)...")

# Definimos el espacio de búsqueda
param_grid = {
    'max_depth': [10, 15, 20],               # Probamos distintas profundidades
    'min_samples_split': [100, 200, 300],    # Diferentes límites para dividir nodos
    'class_weight': [None, 'balanced'],      # Probamos si balancear las clases ayuda
    'criterion': ['gini', 'entropy']         # Diferentes formas de medir la impureza
}

# Usamos cv=3 (3-fold cross-validation) para que sea rápido
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',       # Le decimos que nos importa maximizar el F1 Score
    n_jobs=-1,          # Usa todos los procesadores disponibles
    verbose=1           # Muestra el progreso en la terminal
)

start_time = time.time()

# Ajustamos la búsqueda completa con tu train
grid_search.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

# Extraemos el mejor modelo que encontró
best_dt = grid_search.best_estimator_

print("\n¡Búsqueda terminada!")
print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")

# ---------------------------------------------------------
# 8. EVALUAR EN VALIDATION (CON EL MEJOR MODELO)
# ---------------------------------------------------------
print("\nEvaluando el mejor modelo en validation.csv...")
y_pred = best_dt.predict(X_val_scaled)

# ---------------------------------------------------------
# 9. REGISTRAR RESULTADOS
# ---------------------------------------------------------
acc = accuracy_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print("-" * 40)
print("RESULTADOS PARA EL SHEET (Mejor Decision Tree):")
print(f"Accuracy: {acc:.4f}")
print(f"Recall:   {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Training Time Total: {training_time:.2f} segundos")
print("-" * 40)

# Guardar predicciones
predictions_df = pd.DataFrame({
    'observation_id': val_ids,
    'prediction': y_pred
})
predictions_df.to_csv('dt_optimized_predictions.csv', index=False)
print("Nuevas predicciones guardadas en 'dt_optimized_predictions.csv'")

Cargando datasets...
Aplicando preprocesamiento...
Iniciando búsqueda de hiperparámetros (esto tomará un par de minutos)...
Fitting 3 folds for each of 36 candidates, totalling 108 fits

¡Búsqueda terminada!
Mejores hiperparámetros encontrados:
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 200}

Evaluando el mejor modelo en validation.csv...
----------------------------------------
RESULTADOS PARA EL SHEET (Mejor Decision Tree):
Accuracy: 0.6964
Recall:   0.7112
F1 Score: 0.7129
Training Time Total: 716.90 segundos
----------------------------------------
Nuevas predicciones guardadas en 'dt_optimized_predictions.csv'


In [10]:
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score

# ---------------------------------------------------------
# 1. CARGAR DATOS
# ---------------------------------------------------------
print("Cargando datasets...")
train = pd.read_csv('train_200000.csv', header=None)
val = pd.read_csv('validation.csv', header=None)

# ---------------------------------------------------------
# 2. SEPARAR X y y (Columna 0 es target, 1:29 características)
# ---------------------------------------------------------
y_train = train.iloc[:, 0]
X_train = train.iloc[:, 1:29]

y_val = val.iloc[:, 0]
X_val = val.iloc[:, 1:29]

if 'observation_id' in val.columns:
    val_ids = val['observation_id']
    X_val = X_val.drop(columns=['observation_id'], errors='ignore')
    X_train = X_train.drop(columns=['observation_id'], errors='ignore')
else:
    val_ids = val.index

# ---------------------------------------------------------
# 3. PREPROCESSING (Regla estricta de Edgar)
# ---------------------------------------------------------
print("Aplicando StandardScaler (fit solo en train)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# ---------------------------------------------------------
# 4. ENTRENAR RANDOM FOREST
# ---------------------------------------------------------
print("Entrenando Random Forest (esto tomará un momento)...")
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=150,          # 150 árboles en el bosque
    max_depth=15,              # Profundidad controlada para evitar sobreajuste
    min_samples_split=50,      # Muestras mínimas para dividir
    class_weight='balanced',   # Ayuda a mejorar el Recall
    n_jobs=-1,                 # Usa todos los procesadores para máxima velocidad
    random_state=42            # Semilla para reproducibilidad
)

rf_model.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

# ---------------------------------------------------------
# 5. EVALUAR EN VALIDATION
# ---------------------------------------------------------
print("Evaluando modelo...")
y_pred = rf_model.predict(X_val_scaled)

# ---------------------------------------------------------
# 6. REGISTRAR RESULTADOS
# ---------------------------------------------------------
acc = accuracy_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print("-" * 40)
print("RESULTADOS PARA EL SHEET (Random Forest):")
print(f"Accuracy: {acc:.4f}")
print(f"Recall:   {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Training Time: {training_time:.2f} segundos")
print("-" * 40)

# Guardar las predicciones para Edgar
predictions_df = pd.DataFrame({
    'observation_id': val_ids,
    'prediction': y_pred
})
predictions_df.to_csv('rf_validation_predictions.csv', index=False)
print("Predicciones guardadas en 'rf_validation_predictions.csv'")

Cargando datasets...
Aplicando StandardScaler (fit solo en train)...
Entrenando Random Forest (esto tomará un momento)...
Evaluando modelo...
----------------------------------------
RESULTADOS PARA EL SHEET (Random Forest):
Accuracy: 0.7227
Recall:   0.7169
F1 Score: 0.7326
Training Time: 188.31 segundos
----------------------------------------
Predicciones guardadas en 'rf_validation_predictions.csv'


In [12]:
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, recall_score, f1_score

# ---------------------------------------------------------
# 1. CARGAR DATOS
# ---------------------------------------------------------
print("Cargando datasets...")
train = pd.read_csv('train_200000.csv', header=None)
val = pd.read_csv('validation.csv', header=None)

y_train = train.iloc[:, 0]
X_train = train.iloc[:, 1:29]

y_val = val.iloc[:, 0]
X_val = val.iloc[:, 1:29]

if 'observation_id' in val.columns:
    val_ids = val['observation_id']
    X_val = X_val.drop(columns=['observation_id'], errors='ignore')
    X_train = X_train.drop(columns=['observation_id'], errors='ignore')
else:
    val_ids = val.index

# ---------------------------------------------------------
# 2. PREPROCESSING ESTRICTO: TRANSFORMACIÓN POLINOMIAL
# ---------------------------------------------------------
print("Creando características polinomiales (Grado 2)...")
# include_bias=False evita crear una columna de puros unos que no aporta al modelo
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)

# FIT y TRANSFORM solo en Train
X_train_poly = poly.fit_transform(X_train)
# SOLO TRANSFORM en Validation
X_val_poly = poly.transform(X_val)

print(f"¡Las variables originales pasaron de {X_train.shape[1]} a {X_train_poly.shape[1]} dimensiones!")

# ---------------------------------------------------------
# 3. PREPROCESSING ESTRICTO: ESCALADO
# ---------------------------------------------------------
print("Aplicando StandardScaler al nuevo espacio dimensional...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_val_scaled = scaler.transform(X_val_poly)

# ---------------------------------------------------------
# 4. ENTRENAR REGRESIÓN LOGÍSTICA
# ---------------------------------------------------------
print("Entrenando Regresión Logística (L-BFGS)...")
start_time = time.time()

lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=3000,             # Límite aumentado por la alta dimensionalidad
    class_weight='balanced',
    C=0.1,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

# ---------------------------------------------------------
# 5. EVALUAR EN VALIDATION
# ---------------------------------------------------------
print("Evaluando modelo...")
y_pred = lr_model.predict(X_val_scaled)

acc = accuracy_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print("-" * 40)
print("RESULTADOS PARA EL SHEET (Logistic Regression Poly):")
print(f"Accuracy: {acc:.4f}")
print(f"Recall:   {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Training Time: {training_time:.2f} segundos")
print("-" * 40)

# Guardar predicciones
predictions_df = pd.DataFrame({
    'observation_id': val_ids,
    'prediction': y_pred
})
predictions_df.to_csv('lr_poly_validation_predictions.csv', index=False)
print("Predicciones guardadas en 'lr_poly_validation_predictions.csv'")

Cargando datasets...
Creando características polinomiales (Grado 2)...
¡Las variables originales pasaron de 28 a 434 dimensiones!
Aplicando StandardScaler al nuevo espacio dimensional...
Entrenando Regresión Logística (L-BFGS)...
Evaluando modelo...
----------------------------------------
RESULTADOS PARA EL SHEET (Logistic Regression Poly):
Accuracy: 0.6813
Recall:   0.7172
F1 Score: 0.7046
Training Time: 65.90 segundos
----------------------------------------
Predicciones guardadas en 'lr_poly_validation_predictions.csv'
